## Check PBL Height from ARCO ERA5

Quick utility to check the maximum planetary boundary layer height over a given time range and bounding box. Use this to inform what `DEFAULT_ZG_TOP_PA` should be set to — the top of the control volume should be above the PBL.

In [1]:
%pip install -q zarr gcsfs dask xarray numpy

In [ ]:
import numpy as np
import xarray as xr

import gcsfs
import zarr

import dask.array as da
import pandas as pd

In [6]:
# --- Configuration (edit these) ---
TIME_START = "1941-06-01T00:00:00"
TIME_END   = "1941-06-30T00:00:00"
BBOX       = (40, 60, -130, -110)  # (lat_min, lat_max, lon_min, lon_max)

ARCO_PATH  = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

In [ ]:
lat_min, lat_max, lon_min, lon_max = BBOX

# Open the zarr store directly and read only the PBL array + its coords.
fs = gcsfs.GCSFileSystem(token="anon")
store = gcsfs.GCSMap(ARCO_PATH, gcs=fs)
root = zarr.open_consolidated(store, mode="r")

# Read coordinate arrays
time_raw = root["time"][:]
lat_name = "latitude" if "latitude" in root else "lat"
lon_name = "longitude" if "longitude" in root else "lon"
lat_coord = root[lat_name][:]
lon_coord = root[lon_name][:]

# Debug: inspect time encoding
print(f"time dtype: {time_raw.dtype}, first 3 values: {time_raw[:3]}")
time_attrs = dict(root["time"].attrs)
print(f"time attrs: {time_attrs}")

# Decode time using zarr metadata (units + calendar)
if "units" in time_attrs:
    time_coord = xr.coding.times.decode_cf_datetime(time_raw, time_attrs["units"], time_attrs.get("calendar", "standard"))
elif np.issubdtype(time_raw.dtype, np.datetime64):
    time_coord = time_raw
elif np.issubdtype(time_raw.dtype, np.integer):
    time_coord = pd.to_datetime(time_raw, unit="ns")
else:
    time_coord = pd.to_datetime(time_raw)

print(f"Decoded time range: {time_coord[0]} to {time_coord[-1]}")

# Normalise longitudes to [-180, 180]
if lon_coord.max() > 180:
    lon_coord = (lon_coord + 180) % 360 - 180

# Build lazy DataArray
pbl_zarr = root["boundary_layer_height"]
pbl = xr.DataArray(
    da.from_zarr(pbl_zarr),
    dims=("time", lat_name, lon_name),
    coords={"time": time_coord, lat_name: lat_coord, lon_name: lon_coord},
    name="boundary_layer_height",
)

# Standardize coord names
if lat_name != "lat":
    pbl = pbl.rename({lat_name: "lat"})
if lon_name != "lon":
    pbl = pbl.rename({lon_name: "lon"})

# Sort so slicing works correctly
pbl = pbl.sortby("lat").sortby("lon")

print("Loaded boundary_layer_height array")

pbl = pbl.sel(
    time=slice(TIME_START, TIME_END),
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max),
)

print(f"Time range : {TIME_START} to {TIME_END}")
print(f"Bbox       : lat [{lat_min}, {lat_max}], lon [{lon_min}, {lon_max}]")
print(f"Grid points: time={pbl.sizes.get('time', '?')}, lat={pbl.sizes.get('lat', '?')}, lon={pbl.sizes.get('lon', '?')}")

In [ ]:
print("Computing PBL statistics...")

pbl_max  = float(pbl.max().compute())
pbl_p99  = float(pbl.quantile(0.99).compute())
pbl_p95  = float(pbl.quantile(0.95).compute())
pbl_mean = float(pbl.mean().compute())

print(f"\n--- PBL height statistics [m] ---")
print(f"  Max  : {pbl_max:>10.1f} m")
print(f"  P99  : {pbl_p99:>10.1f} m")
print(f"  P95  : {pbl_p95:>10.1f} m")
print(f"  Mean : {pbl_mean:>10.1f} m")

# --- Estimate pressure at PBL top using ERA5 geopotential ---
# geopotential Φ [m²/s²]; geopotential height Z = Φ / g [m]
g = 9.806  # m/s², consistent with project config

print("\nLoading geopotential field...")
geo_zarr = root["geopotential"]  # (time, level, lat, lon)

# Read coordinate arrays for the 3D field
level_coord = root["pressure_level"][:]  # pressure levels [hPa in ARCO]
level_pa = level_coord * 100.0  # convert to Pa

# Build lazy DataArray for geopotential
geo = xr.DataArray(
    da.from_zarr(geo_zarr),
    dims=("time", "level", lat_name, lon_name),
    coords={
        "time": time_coord,
        "level": level_pa,
        lat_name: lat_coord,
        lon_name: lon_coord,
    },
    name="geopotential",
)
if lat_name != "lat":
    geo = geo.rename({lat_name: "lat"})
if lon_name != "lon":
    geo = geo.rename({lon_name: "lon"})
geo = geo.sortby("lat").sortby("lon")

geo = geo.sel(
    time=slice(TIME_START, TIME_END),
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max),
)

# Convert to geopotential height [m]
Z = geo / g

print(f"Geopotential grid: time={Z.sizes['time']}, level={Z.sizes['level']}, lat={Z.sizes['lat']}, lon={Z.sizes['lon']}")
print("Computing geopotential (this may take a moment)...")
Z = Z.compute()

# Pressure coordinate [Pa]
p_levels = Z.coords["level"].values
ln_p = np.log(p_levels)

def pressure_at_height(z_target, Z_field, ln_p_arr):
    """Interpolate to find pressure where geopotential height = z_target.

    Uses log-pressure interpolation: interp ln(p) as a function of Z.
    Z decreases with increasing pressure, so we flip to make Z ascending.
    Returns the minimum pressure (most conservative) across the domain.
    """
    # Flip level axis so Z is ascending
    Z_vals = Z_field.values[:, ::-1, :, :]  # (time, level, lat, lon)
    lnp_asc = ln_p_arr[::-1]  # ln(p) now descending (low pressure first)

    n_time, n_lev, n_lat, n_lon = Z_vals.shape
    Z_flat = Z_vals.reshape(n_time, n_lev, -1)
    n_col = Z_flat.shape[2]

    result = np.full((n_time, n_col), np.nan)
    for t in range(n_time):
        for c in range(n_col):
            z_col = Z_flat[t, :, c]
            if np.all(np.isnan(z_col)):
                continue
            result[t, c] = np.interp(z_target, z_col, lnp_asc,
                                     left=np.nan, right=np.nan)

    return np.exp(result)

p_at_max_arr = pressure_at_height(pbl_max, Z, ln_p)
p_at_p99_arr = pressure_at_height(pbl_p99, Z, ln_p)
p_at_p95_arr = pressure_at_height(pbl_p95, Z, ln_p)

# Most conservative: minimum pressure (highest altitude) across domain
p_at_max = float(np.nanmin(p_at_max_arr))
p_at_p99 = float(np.nanmin(p_at_p99_arr))
p_at_p95 = float(np.nanmin(p_at_p95_arr))

print(f"\n--- Pressure at PBL top (from ERA5 geopotential) [Pa] ---")
print(f"  At max PBL : {p_at_max:>10.0f} Pa  ({p_at_max/100:>7.1f} hPa)")
print(f"  At P99 PBL : {p_at_p99:>10.0f} Pa  ({p_at_p99/100:>7.1f} hPa)")
print(f"  At P95 PBL : {p_at_p95:>10.0f} Pa  ({p_at_p95/100:>7.1f} hPa)")

print(f"\n--- Recommendation ---")
print(f"  Set DEFAULT_ZG_TOP_PA to at most {round(p_at_max / 100) * 100:.0f} Pa")
print(f"  (rounded down from {p_at_max:.0f} Pa at the max PBL height)")
print(f"  This ensures the control volume top is above the PBL at all times.")